## 07b_lenormand_pytdlm_pl
  Same as 07 for PL NUTS3, both small-flow variants.
  Output: data/processed/07b_lenormand_pytdlm_pl/. Environment: nuts3_tdlm.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "PL"
COL_ID, COL_POP = "nuts3_id", "population"
SHARED_DIR = config.INTERIM_DIR / "lenormand_comparison" / COUNTRY.lower()
POP_CSV = SHARED_DIR / "population.csv"
DIST_CSV = SHARED_DIR / "distance_km.csv"
OD_PREFIX = "od_observed_"                          # files: od_observed_<variant>.csv
OUT_DIR = config.PROCESSED_DIR / "07b_lenormand_pytdlm_pl"
# ------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

pop = pd.read_csv(POP_CSV)
dist_df = pd.read_csv(DIST_CSV, index_col=0)
ids = pop[COL_ID].tolist()
assert list(dist_df.index) == ids == list(dist_df.columns), "distance matrix order differs"

mass = pop[COL_POP].to_numpy(dtype=float)
distance = dist_df.to_numpy(dtype=float)

# Variants are read from the file names in the shared folder
od_files = sorted(SHARED_DIR.glob(f"{OD_PREFIX}*.csv"))
VARIANTS = [p.stem.removeprefix(OD_PREFIX) for p in od_files]

OBS, OUT_TRIPS, IN_TRIPS = {}, {}, {}
for variant, path in zip(VARIANTS, od_files):
    od_df = pd.read_csv(path, index_col=0)
    assert list(od_df.index) == ids == list(od_df.columns), "OD matrix order differs"
    obs = od_df.to_numpy(dtype=float).copy()
    np.fill_diagonal(obs, 0)                      # T_ii = 0, as in the paper
    OBS[variant] = obs
    OUT_TRIPS[variant] = obs.sum(axis=1)          # O_i
    IN_TRIPS[variant] = obs.sum(axis=0)           # D_j
    assert np.isclose(OUT_TRIPS[variant].sum(), IN_TRIPS[variant].sum()), "marginals differ"
    assert (OUT_TRIPS[variant] > 0).all() and (IN_TRIPS[variant] > 0).all(), "zero marginal"

print("Regions:", len(ids))
print("Shapes: mass", mass.shape, "| distance", distance.shape)
for variant in VARIANTS:
    print(f"{variant}: obs {OBS[variant].shape} | total {int(OBS[variant].sum())} "
          f"| positive pairs {int((OBS[variant] > 0).sum())}")
print("Loaded from:", os.path.relpath(SHARED_DIR))

In [ ]:
import io
import warnings
from contextlib import redirect_stdout
from TDLM import tdlm

# ----------------------------- CONFIG -----------------------------
CALIB_CSV = OUT_DIR / "calibrated_exponents.csv"
CALIB_LAWS = ["NGravExp", "NGravPow"]               # Rad has no parameter
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
N_REPS_CALIB = 10
SEED = 0
CALIB_MEASURE = "CPC"
# VARIANTS, OBS, OUT_TRIPS, IN_TRIPS, mass, distance come from the previous cell
# ------------------------------------------------------------------

warnings.filterwarnings("ignore", category=RuntimeWarning)

if CALIB_CSV.exists():
    calib = pd.read_csv(CALIB_CSV)
    print("Loaded from checkpoint:", os.path.relpath(CALIB_CSV))
else:
    frames = []
    for variant in VARIANTS:
        with redirect_stdout(io.StringIO()):        # silence the package's own progress prints
            res = tdlm.run_optimization(
                mass_origin=mass, mass_destination=mass, distance=distance, obs=OBS[variant],
                law=CALIB_LAWS, model=CONSTRAINT_LEVELS,
                out_trips=OUT_TRIPS[variant], in_trips=IN_TRIPS[variant],
                repli=N_REPS_CALIB, measure=CALIB_MEASURE, random_seed=SEED)
        res.insert(0, "Variant", variant)
        frames.append(res)
        print("Done:", variant)
    calib = pd.concat(frames, ignore_index=True)
    calib.to_csv(CALIB_CSV, index=False)
    print("Saved:", os.path.relpath(CALIB_CSV))

print(calib.round(4).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------- CONFIG -----------------------------
N_REPS = 100
SEED_EVAL = 1
LAWS = ["NGravExp", "NGravPow", "Rad", "Rand"]
PARAMETER_FREE = ("Rad", "Rand")                    # no exponent to calibrate
METRIC_COLS = ["CPC", "CPL", "CPCd", "KL_div", "RMSE"]    # KS columns are not used
RUNS_DIR = OUT_DIR / "runs"
# VARIANTS, CONSTRAINT_LEVELS, calib, OBS, OUT_TRIPS, IN_TRIPS, mass, distance come from the earlier cells
# ------------------------------------------------------------------

RUNS_DIR.mkdir(parents=True, exist_ok=True)

def first_value(x):
    """Return the object itself or the first item of a dict keyed by exponent."""
    return next(iter(x.values())) if isinstance(x, dict) else x

for variant in VARIANTS:
    for law in LAWS:
        for level in CONSTRAINT_LEVELS:
            tag = f"{variant}_{law}_{level}"
            sim_path, met_path = RUNS_DIR / f"sim_{tag}.npy", RUNS_DIR / f"metrics_{tag}.csv"
            if sim_path.exists() and met_path.exists():
                print("Skip (checkpoint):", tag)
                continue
            if law in PARAMETER_FREE:
                exponent = 1.0                       # ignored by parameter-free laws
            else:
                row = calib[(calib["Variant"] == variant) & (calib["Law"] == law)
                            & (calib["Model"] == level)].iloc[0]
                exponent = np.array([row["Exponent"]])
            with redirect_stdout(io.StringIO()):     # silence the package's own progress prints
                sim = tdlm.run_law_model(
                    law=law, mass_origin=mass, mass_destination=mass, distance=distance,
                    exponent=exponent, model=level,
                    out_trips=OUT_TRIPS[variant], in_trips=IN_TRIPS[variant],
                    repli=N_REPS, random_seed=SEED_EVAL, verbose=False)
                res = tdlm.gof(sim=sim, obs=OBS[variant], distance=distance)
            met = first_value(res)[["Replication"] + METRIC_COLS]
            np.save(sim_path, np.asarray(first_value(sim)))
            met.to_csv(met_path, index=False)
            print("Done:", tag, "| CPC mean:", round(met["CPC"].mean(), 4))

In [ ]:
# ----------------------------- CONFIG -----------------------------
BY_REP_CSV = OUT_DIR / "metrics_pytdlm_by_rep.csv"
SUMMARY_CSV = OUT_DIR / "metrics_pytdlm_summary.csv"
# VARIANTS, LAWS, CONSTRAINT_LEVELS, METRIC_COLS, RUNS_DIR come from the earlier cells
# ------------------------------------------------------------------

frames = []
for variant in VARIANTS:
    for law in LAWS:
        for level in CONSTRAINT_LEVELS:
            m = pd.read_csv(RUNS_DIR / f"metrics_{variant}_{law}_{level}.csv")
            m.insert(0, "Model", level)
            m.insert(0, "Law", law)
            m.insert(0, "Variant", variant)
            frames.append(m)
by_rep = pd.concat(frames, ignore_index=True)

summary = (by_rep.groupby(["Variant", "Law", "Model"], sort=False)[METRIC_COLS]
           .agg(["mean", "std"]))
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary = summary.reset_index()

by_rep.to_csv(BY_REP_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)
print("Saved:", os.path.relpath(BY_REP_CSV), by_rep.shape)
print("Saved:", os.path.relpath(SUMMARY_CSV), summary.shape)
print(summary[["Variant", "Law", "Model", "CPC_mean", "CPL_mean", "CPCd_mean", "KL_div_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
# ----------------------------- CONFIG -----------------------------
SKMOB_DIR = config.PROCESSED_DIR / "06b_lenormand_skmob_pl"
SKMOB_SUMMARY_CSV = SKMOB_DIR / "metrics_skmob_summary.csv"
XCHECK_BY_REP_CSV = OUT_DIR / "metrics_skmob_in_pytdlm_by_rep.csv"
XCHECK_SUMMARY_CSV = OUT_DIR / "metrics_skmob_in_pytdlm_summary.csv"
# OBS, distance, METRIC_COLS, first_value, tdlm come from the earlier cells
# ------------------------------------------------------------------

skmob_summary = pd.read_csv(SKMOB_SUMMARY_CSV)

if XCHECK_BY_REP_CSV.exists() and XCHECK_SUMMARY_CSV.exists():
    xs = pd.read_csv(XCHECK_SUMMARY_CSV)
    print("Loaded from checkpoint:", os.path.relpath(XCHECK_SUMMARY_CSV))
else:
    frames = []
    for r in skmob_summary.itertuples():
        sim = np.load(SKMOB_DIR / f"realizations_{r.variant}_{r.model}.npy")   # (repli, n, n)
        with redirect_stdout(io.StringIO()):
            res = tdlm.gof(sim=sim, obs=OBS[r.variant], distance=distance)
        m = first_value(res)[["Replication"] + METRIC_COLS].copy()
        m.insert(0, "SkmobModel", r.model)
        m.insert(0, "Variant", r.variant)
        frames.append(m)
    xrep = pd.concat(frames, ignore_index=True)
    xs = (xrep.groupby(["Variant", "SkmobModel"], sort=False)[METRIC_COLS].agg(["mean", "std"]))
    xs.columns = [f"{a}_{b}" for a, b in xs.columns]
    xs = xs.reset_index()
    # add skmob's own CPC and the difference (cross-check)
    xs = xs.merge(skmob_summary[["variant", "model", "cpc_mean", "ig_mean_finite"]]
                  .rename(columns={"variant": "Variant", "model": "SkmobModel",
                                   "cpc_mean": "CPC_skmob", "ig_mean_finite": "InfoGain_skmob"}),
                  on=["Variant", "SkmobModel"])
    xs["CPC_diff"] = xs["CPC_mean"] - xs["CPC_skmob"]
    xrep.to_csv(XCHECK_BY_REP_CSV, index=False)
    xs.to_csv(XCHECK_SUMMARY_CSV, index=False)
    print("Saved:", os.path.relpath(XCHECK_BY_REP_CSV), xrep.shape)
    print("Saved:", os.path.relpath(XCHECK_SUMMARY_CSV), xs.shape)

print(xs[["Variant", "SkmobModel", "CPC_mean", "CPC_skmob", "CPC_diff",
          "KL_div_mean", "InfoGain_skmob", "CPL_mean", "CPCd_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG_NAME = "pl_lenormand_cpc_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
LAW_STYLE = {                        # color, marker, legend label
    "NGravExp": ("#2a78d6", "o", "NGrav (exp)"),
    "NGravPow": ("#eb6834", "s", "NGrav (pow)"),
    "Rad":      ("#1baf7a", "^", "Rad"),
}
LAW_Y_OFFSET = {"NGravExp": 0.2, "NGravPow": 0.0, "Rad": -0.2}
FILLED_VARIANT = VARIANTS[0]         # drawn with filled marks, the others with open marks
X_PAD = 0.03
# summary, VARIANTS, CONSTRAINT_LEVELS come from the earlier cells
# ------------------------------------------------------------------

def variant_value(variant):
    """Fill value of the unobserved pairs, read from the variant name (e.g. missing2 -> 2)."""
    return int(variant.removeprefix("missing"))

def variant_label(variant):
    return f"Unobserved pairs (T < 3) = {variant_value(variant)}"

ypos = {lvl: len(CONSTRAINT_LEVELS) - i for i, lvl in enumerate(CONSTRAINT_LEVELS)}   # UM on top
cpc = summary[summary["Law"].isin(LAW_STYLE)]
xmin, xmax = cpc["CPC_mean"].min() - X_PAD, cpc["CPC_mean"].max() + X_PAD

fig, ax = plt.subplots(figsize=(9, 4.8))
for variant in VARIANTS:
    filled = variant == FILLED_VARIANT
    for law, (color, marker, label) in LAW_STYLE.items():
        sub = cpc[(cpc["Variant"] == variant) & (cpc["Law"] == law)]
        ax.scatter(sub["CPC_mean"], [ypos[m] + LAW_Y_OFFSET[law] for m in sub["Model"]],
                   marker=marker, s=60 if filled else 110,
                   facecolor=color if filled else "none",
                   edgecolor="white" if filled else color,
                   linewidth=1.2 if filled else 1.8, zorder=3 if filled else 4)
ax.set_yticks(list(ypos.values()))
ax.set_yticklabels(list(ypos.keys()))
ax.set_xlim(xmin, xmax)
ax.set_xlabel("CPC (mean of replications)")
ax.set_title("PL NUTS3 level: CPC by law and constraint level (PyTDLM)", fontweight="bold")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)

law_handles = [Line2D([], [], linestyle="none", marker=m, markersize=8, markerfacecolor=c,
                      markeredgecolor="white", label=l) for c, m, l in LAW_STYLE.values()]
variant_handles = [Line2D([], [], linestyle="none", marker="o", markersize=8,
                          markerfacecolor="#555555" if v == FILLED_VARIANT else "none",
                          markeredgecolor="#555555", label=variant_label(v)) for v in VARIANTS]
fig.legend(handles=law_handles + variant_handles, loc="lower center", ncol=5, frameon=False,
           fontsize=9, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.1, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG4_NAME = "pl_lenormand_fig4_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
PAPER_STYLE = {                  # facecolor, marker, legend label (as in the paper)
    "NGravExp": ("#ee6a5b", "o", "NGrav (exp)"),
    "NGravPow": ("#4b8bea", "s", "NGrav (pow)"),
    "Rad":      ("#a94fc0", "^", "Rad"),
}
PANELS = [("(a)", "CPC", "CPC"), ("(b)", "CPL", "CPL"), ("(c)", "CPCd", r"CPC$_d$")]
RAND_LAW = "Rand"
RAND_STYLE = {                   # metric -> (color, linestyle, label)
    "CPC": ("black", "-", "CPC"),
    "CPL": ("#666666", "--", "CPL"),
    "CPCd": ("#999999", ":", r"CPC$_d$"),
}
OVERLAY_METRICS = {"CPL"}        # metrics where both variants are drawn (the others differ by <= 0.002)
HEADROOM = 0.15                  # empty space above the data
# summary, VARIANTS, CONSTRAINT_LEVELS, FILLED_VARIANT, variant_label come from the earlier cells
# ------------------------------------------------------------------

x = list(range(len(CONSTRAINT_LEVELS)))

def variants_for(metric):
    """All variants for the overlay metrics, only the first (filled) variant otherwise."""
    return VARIANTS if metric in OVERLAY_METRICS else [FILLED_VARIANT]

def pad_ylim(ax, values):
    lo, hi = min(values), max(values)
    span = (hi - lo) or 0.1
    ax.set_ylim(lo - 0.1 * span, hi + HEADROOM * span)

def style_axis(ax, title, ylabel):
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CONSTRAINT_LEVELS, fontweight="bold")
    ax.set_xlabel("Constrained Models")
    ax.set_ylabel(ylabel)

def series(law, metric, variant):
    sub = summary[(summary["Variant"] == variant) & (summary["Law"] == law)].set_index("Model")
    return sub.loc[CONSTRAINT_LEVELS, f"{metric}_mean"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))
for ax, (title, metric, ylabel) in zip(axes.flat[:3], PANELS):
    vals = []
    for variant in variants_for(metric):
        filled = variant == FILLED_VARIANT
        for law, (color, marker, label) in PAPER_STYLE.items():
            y = series(law, metric, variant)
            vals += list(y)
            ax.plot(x, y, color="black", linewidth=1, linestyle="-" if filled else "--", zorder=2)
            ax.plot(x, y, linestyle="none", marker=marker, markersize=8 if filled else 11,
                    markerfacecolor=color if filled else "none",
                    markeredgecolor="black" if filled else color,
                    markeredgewidth=1 if filled else 1.8, zorder=3 if filled else 4)
    style_axis(ax, title, ylabel)
    pad_ylim(ax, vals)

# panel (d): uniform distribution (Rand law); both variants only for the CPL line
ax = axes[1, 1]
vals = []
for metric, (color, ls, label) in RAND_STYLE.items():
    for variant in variants_for(metric):
        filled = variant == FILLED_VARIANT
        y = series(RAND_LAW, metric, variant)
        vals += list(y)
        ax.plot(x, y, color=color, linestyle=ls if filled else (0, (2, 2)), linewidth=1.3,
                marker="o", markersize=6 if filled else 9,
                markerfacecolor=color if filled else "none", markeredgecolor=color,
                markeredgewidth=1 if filled else 1.5)
style_axis(ax, "(d)", "Random")
pad_ylim(ax, vals)
metric_handles = [Line2D([], [], color=c, linestyle=ls, marker="o", markersize=6, label=l)
                  for c, ls, l in RAND_STYLE.values()]
ax.legend(handles=metric_handles, loc="center left", frameon=True, fontsize=9)

# one shared legend below: laws and the two variants
law_handles = [Line2D([], [], linestyle="none", marker=m, markersize=8, markerfacecolor=c,
                      markeredgecolor="black", label=l) for c, m, l in PAPER_STYLE.values()]
variant_handles = [Line2D([], [], color="#555555", linestyle="-" if v == FILLED_VARIANT else "--",
                          marker="o", markersize=7,
                          markerfacecolor="#555555" if v == FILLED_VARIANT else "none",
                          markeredgecolor="#555555", label=variant_label(v)) for v in VARIANTS]
fig.legend(handles=law_handles + variant_handles, loc="lower center", ncol=5, frameon=False,
           fontsize=9, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.05, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG4_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG4_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
LINKS_CSV = OUT_DIR / "links_ratio.csv"
FIG5_NAME = "pl_lenormand_fig5_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
LINK_LAWS = list(PAPER_STYLE)                          # NGravExp, NGravPow, Rad
PANEL_TITLES = {"UM": "Unconstrained Model", "PCM": "Production Constrained Model",
                "ACM": "Attraction Constrained Model", "DCM": "Doubly Constrained Model"}
LAW_Y_OFFSET = {"NGravExp": 0.2, "NGravPow": 0.0, "Rad": -0.2}   # separate the laws vertically
# RUNS_DIR, OBS, VARIANTS, CONSTRAINT_LEVELS, PAPER_STYLE, FILLED_VARIANT, variant_label
# come from the earlier cells
# ------------------------------------------------------------------

n = len(ids)
offdiag = ~np.eye(n, dtype=bool)

if LINKS_CSV.exists():
    links = pd.read_csv(LINKS_CSV)
    print("Loaded from checkpoint:", os.path.relpath(LINKS_CSV))
else:
    rows = []
    for variant in VARIANTS:
        obs_links = int(((OBS[variant] > 0) & offdiag).sum())
        for law in LINK_LAWS:
            for level in CONSTRAINT_LEVELS:
                sim = np.load(RUNS_DIR / f"sim_{variant}_{law}_{level}.npy")   # (repli, n, n)
                ratio = ((sim > 0) & offdiag).sum(axis=(1, 2)) / obs_links      # one value per replication
                rows.append({"Variant": variant, "Law": law, "Model": level, "obs_links": obs_links,
                             "ratio_mean": ratio.mean(), "ratio_min": ratio.min(), "ratio_max": ratio.max()})
    links = pd.DataFrame(rows)
    links.to_csv(LINKS_CSV, index=False)
    print("Saved:", os.path.relpath(LINKS_CSV), links.shape)
print(links.round(4).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(9.5, 5.8), sharex=True)
for ax, level in zip(axes.flat, CONSTRAINT_LEVELS):
    ax.axvline(1.0, color="black", linewidth=1, zorder=1)
    for variant in VARIANTS:
        filled = variant == FILLED_VARIANT
        for law in LINK_LAWS:
            color, marker, label = PAPER_STYLE[law]
            r = links[(links["Variant"] == variant) & (links["Law"] == law)
                      & (links["Model"] == level)].iloc[0]
            y = LAW_Y_OFFSET[law]
            # clip at 0: rounding can make mean - min a tiny negative number
            err_low = max(r["ratio_mean"] - r["ratio_min"], 0.0)
            err_high = max(r["ratio_max"] - r["ratio_mean"], 0.0)
            ax.errorbar(r["ratio_mean"], y, xerr=[[err_low], [err_high]],
                    color="black" if filled else color, linewidth=1, capsize=0, zorder=2)
            ax.plot(r["ratio_mean"], y, linestyle="none", marker=marker,
                    markersize=9 if filled else 12,
                    markerfacecolor=color if filled else "none",
                    markeredgecolor="black" if filled else color,
                    markeredgewidth=1 if filled else 1.8, zorder=3 if filled else 4)
    ax.set_title(PANEL_TITLES[level], fontweight="bold")
    ax.set_yticks([0])
    ax.set_yticklabels([COUNTRY], fontweight="bold")
    ax.set_ylim(-0.5, 0.5)
x_max = float(links["ratio_max"].max())
for ax in axes.flat:
    ax.set_xlim(0, max(1.1, x_max * 1.05))
for ax in axes[1]:
    ax.set_xlabel("(#Simulated links) / (#Observed links)")

law_handles = [Line2D([], [], linestyle="none", marker=m, markersize=8, markerfacecolor=c,
                      markeredgecolor="black", label=l) for c, m, l in PAPER_STYLE.values()]
variant_handles = [Line2D([], [], linestyle="none", marker="o", markersize=8,
                          markerfacecolor="#555555" if v == FILLED_VARIANT else "none",
                          markeredgecolor="#555555", label=variant_label(v)) for v in VARIANTS]
fig.legend(handles=law_handles + variant_handles, loc="lower center", ncol=5, frameon=False,
           fontsize=9, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.07, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG5_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG5_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG6_LEVEL = "PCM"                      # the paper shows the production constrained model
FIG6_VARIANT = FILLED_VARIANT           # one variant only (the totals differ by about 0.2%)
BIN_RATIO = 1.2                         # each bin is this many times wider than the previous one
PDF_CSV = OUT_DIR / f"distance_pdf_{FIG6_LEVEL}_logbin.csv"
FIG6_NAME = f"pl_lenormand_fig6_{FIG6_LEVEL}_logbin_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
DATA_MARKER_SIZE, SIM_MARKER_SIZE = 7, 4
# PAPER_STYLE, OBS, distance, RUNS_DIR, COUNTRY come from the earlier cells
# ------------------------------------------------------------------

offdiag = ~np.eye(distance.shape[0], dtype=bool)
d_flat = distance[offdiag]
# geometric bin edges covering the whole distance range
n_bins = int(np.ceil(np.log(d_flat.max() / d_flat.min()) / np.log(BIN_RATIO))) + 1
edges = d_flat.min() * 0.999 * BIN_RATIO ** np.arange(n_bins + 1)
widths = np.diff(edges)
centers = np.sqrt(edges[:-1] * edges[1:])

def distance_pdf(flows):
    """P(d): share of commuters per km in each (log-spaced) distance bin."""
    w = flows[offdiag]
    hist, _ = np.histogram(d_flat, bins=edges, weights=w)
    return hist / w.sum() / widths

pdf = pd.DataFrame({"d_km": centers, "bin_width_km": widths,
                    "Observed": distance_pdf(OBS[FIG6_VARIANT])})
for law in PAPER_STYLE:
    sim = np.load(RUNS_DIR / f"sim_{FIG6_VARIANT}_{law}_{FIG6_LEVEL}.npy")    # (repli, n, n)
    pdf[law] = np.mean([distance_pdf(s) for s in sim], axis=0)
pdf.to_csv(PDF_CSV, index=False)
print("Saved:", os.path.relpath(PDF_CSV), pdf.shape)
print("Non-empty bins (observed):", int((pdf["Observed"] > 0).sum()), "of", len(pdf))

fig, ax = plt.subplots(figsize=(6.5, 5))
obs_nz = pdf[pdf["Observed"] > 0]
ax.plot(obs_nz["d_km"], obs_nz["Observed"], color="black", marker="*", markersize=DATA_MARKER_SIZE,
        linewidth=0.8, label="Data", zorder=4)
for law, (color, marker, label) in PAPER_STYLE.items():
    nz = pdf[pdf[law] > 0]
    ax.plot(nz["d_km"], nz[law], color=color, marker=marker, markersize=SIM_MARKER_SIZE,
            markeredgecolor="black", markeredgewidth=0.5, linewidth=0.8, label=label, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d (km)")
ax.set_ylabel("P(d)")
ax.set_title(f"Commuting distance distribution ({COUNTRY}, {FIG6_LEVEL}, log bins x{BIN_RATIO})",
             fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=4)
fig.tight_layout()

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG6_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG6_NAME))
plt.show()

In [ ]:
# ----------------------------- CONFIG -----------------------------
IO_SUFFIX = "flows"                                   # inputs: O_i and D_j instead of population
RUNS_DIR_IO = OUT_DIR / f"runs_{IO_SUFFIX}"
CALIB_IO_CSV = OUT_DIR / f"calibrated_exponents_{IO_SUFFIX}.csv"
SUMMARY_IO_CSV = OUT_DIR / f"metrics_pytdlm_summary_{IO_SUFFIX}.csv"
IO_LAWS = ["NGravExp", "NGravPow", "Rad"]
# CALIB_LAWS, CONSTRAINT_LEVELS, N_REPS, N_REPS_CALIB, SEED, SEED_EVAL, CALIB_MEASURE,
# METRIC_COLS, PARAMETER_FREE, first_value, VARIANTS, OBS, OUT_TRIPS, IN_TRIPS, distance
# come from the earlier cells
# ------------------------------------------------------------------

RUNS_DIR_IO.mkdir(parents=True, exist_ok=True)

# 1) calibration of the exponents with O_i and D_j as masses
if CALIB_IO_CSV.exists():
    calib_io = pd.read_csv(CALIB_IO_CSV)
    print("Loaded from checkpoint:", os.path.relpath(CALIB_IO_CSV))
else:
    frames = []
    for variant in VARIANTS:
        with redirect_stdout(io.StringIO()):
            res = tdlm.run_optimization(
                mass_origin=OUT_TRIPS[variant], mass_destination=IN_TRIPS[variant],
                distance=distance, obs=OBS[variant],
                law=CALIB_LAWS, model=CONSTRAINT_LEVELS,
                out_trips=OUT_TRIPS[variant], in_trips=IN_TRIPS[variant],
                repli=N_REPS_CALIB, measure=CALIB_MEASURE, random_seed=SEED)
        res.insert(0, "Variant", variant)
        frames.append(res)
        print("Calibrated:", variant)
    calib_io = pd.concat(frames, ignore_index=True)
    calib_io.to_csv(CALIB_IO_CSV, index=False)
    print("Saved:", os.path.relpath(CALIB_IO_CSV))
print(calib_io.round(4).to_string(index=False))

# 2) final runs and goodness-of-fit measures (one checkpoint per model)
frames = []
for variant in VARIANTS:
    for law in IO_LAWS:
        for level in CONSTRAINT_LEVELS:
            tag = f"{variant}_{law}_{level}"
            sim_path, met_path = RUNS_DIR_IO / f"sim_{tag}.npy", RUNS_DIR_IO / f"metrics_{tag}.csv"
            if not (sim_path.exists() and met_path.exists()):
                if law in PARAMETER_FREE:
                    exponent = 1.0                       # ignored by parameter-free laws
                else:
                    row = calib_io[(calib_io["Variant"] == variant) & (calib_io["Law"] == law)
                                   & (calib_io["Model"] == level)].iloc[0]
                    exponent = np.array([row["Exponent"]])
                with redirect_stdout(io.StringIO()):
                    sim = tdlm.run_law_model(
                        law=law, mass_origin=OUT_TRIPS[variant], mass_destination=IN_TRIPS[variant],
                        distance=distance, exponent=exponent, model=level,
                        out_trips=OUT_TRIPS[variant], in_trips=IN_TRIPS[variant],
                        repli=N_REPS, random_seed=SEED_EVAL, verbose=False)
                    res = tdlm.gof(sim=sim, obs=OBS[variant], distance=distance)
                first_value(res)[["Replication"] + METRIC_COLS].to_csv(met_path, index=False)
                np.save(sim_path, np.asarray(first_value(sim)))
                print("Done:", tag)
            else:
                print("Skip (checkpoint):", tag)
            m = pd.read_csv(met_path)
            m.insert(0, "Model", level)
            m.insert(0, "Law", law)
            m.insert(0, "Variant", variant)
            frames.append(m)

by_rep_io = pd.concat(frames, ignore_index=True)
summary_io = by_rep_io.groupby(["Variant", "Law", "Model"], sort=False)[METRIC_COLS].agg(["mean", "std"])
summary_io.columns = [f"{a}_{b}" for a, b in summary_io.columns]
summary_io = summary_io.reset_index()
summary_io.to_csv(SUMMARY_IO_CSV, index=False)
print("Saved:", os.path.relpath(SUMMARY_IO_CSV), summary_io.shape)
print(summary_io[["Variant", "Law", "Model", "CPC_mean", "CPL_mean", "CPCd_mean", "KL_div_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG7_NAME = "pl_lenormand_fig7_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
PANELS7 = [("(a)", "CPC", "CPC"), ("(b)", "CPL", "CPL"), ("(c)", "CPCd", r"CPC$_d$")]
OVERLAY_METRICS = {"CPL"}        # metrics where both variants are drawn
HEADROOM = 0.15
# summary_io, VARIANTS, CONSTRAINT_LEVELS, PAPER_STYLE, FILLED_VARIANT, variant_label
# come from the earlier cells
# ------------------------------------------------------------------

x = list(range(len(CONSTRAINT_LEVELS)))

def series_io(law, metric, variant):
    sub = summary_io[(summary_io["Variant"] == variant) & (summary_io["Law"] == law)].set_index("Model")
    return sub.loc[CONSTRAINT_LEVELS, f"{metric}_mean"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, (title, metric, ylabel) in zip(axes, PANELS7):
    vals = []
    for variant in (VARIANTS if metric in OVERLAY_METRICS else [FILLED_VARIANT]):
        filled = variant == FILLED_VARIANT
        for law, (color, marker, label) in PAPER_STYLE.items():
            y = series_io(law, metric, variant)
            vals += list(y)
            ax.plot(x, y, color="black", linewidth=1, linestyle="-" if filled else "--", zorder=2)
            ax.plot(x, y, linestyle="none", marker=marker, markersize=8 if filled else 11,
                    markerfacecolor=color if filled else "none",
                    markeredgecolor="black" if filled else color,
                    markeredgewidth=1 if filled else 1.8, zorder=3 if filled else 4)
    span = (max(vals) - min(vals)) or 0.1
    ax.set_ylim(min(vals) - 0.1 * span, max(vals) + HEADROOM * span)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CONSTRAINT_LEVELS, fontweight="bold")
    ax.set_xlabel("Constrained Models")
    ax.set_ylabel(ylabel)

law_handles = [Line2D([], [], linestyle="none", marker=m, markersize=8, markerfacecolor=c,
                      markeredgecolor="black", label=l) for c, m, l in PAPER_STYLE.values()]
variant_handles = [Line2D([], [], color="#555555", linestyle="-" if v == FILLED_VARIANT else "--",
                          marker="o", markersize=7,
                          markerfacecolor="#555555" if v == FILLED_VARIANT else "none",
                          markeredgecolor="#555555", label=variant_label(v)) for v in VARIANTS]
fig.legend(handles=law_handles + variant_handles, loc="lower center", ncol=5, frameon=False,
           fontsize=9, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.08, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG7_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG7_NAME))
plt.show()